# Files and data I/O

## Learning objectives

By the end of this notebook you will be able to:

- open a file safely with a `with` block and the right text encoding;
- read and write CSV with the `csv` module and with pandas;
- serialise and restore Python objects as JSON;
- explain the difference between a file *path* and the data it contains;
- clean up temporary artefacts deliberately.

## Concept

Data work begins with reading files and ends with writing them. This notebook covers the three
formats you will meet constantly — plain text via context managers, CSV, and JSON — first with
the standard library and then with pandas. Every example writes to a throwaway temporary
directory, so running the notebook leaves no clutter behind.

A **file path** is just a string (or a `pathlib.Path`) that names a location. The `pathlib`
module makes paths objects that can be joined, tested, and iterated, which is clearer than
manual string concatenation.

A **context manager** is the `with` statement. It guarantees the file is closed even if the code
inside raises, and it makes the scope of the file obvious. Always pass `encoding="utf-8"` for
text so a file written on one machine reads the same on another.

**CSV** stores a table as text: one line per row, commas between fields, and a header line
naming the columns. It is simple and universal but carries no type information — everything
comes back as a string unless something parses it.

**JSON** stores nested data as text using objects, arrays, strings, numbers, booleans, and
`null`. It maps naturally onto Python `dict` and `list`, which makes it the default for APIs
(notebook 05) and configuration.

pandas sits on top of both: `read_csv` / `to_csv` and `read_json` / `to_json` handle types and
indexing for you, at the cost of a little more magic.

## Worked example

### A scratch directory

`tempfile.mkdtemp` hands us a unique directory we can write into freely. We remove it at the end
of the notebook.

In [ ]:
import csv
import json
import shutil
import tempfile
from pathlib import Path

import pandas as pd

from ds_practice import load_penguins

penguins = load_penguins()
workdir = Path(tempfile.mkdtemp(prefix="penguins_io_"))
print("working in:", workdir)
print("exists:", workdir.exists(), "| is a directory:", workdir.is_dir())

### Plain text with a context manager

The `with` block closes the file when the block exits, including on error. Reading it back is a
one-liner.

In [ ]:
note = workdir / "notes.txt"
with note.open("w", encoding="utf-8") as fh:
    fh.write("Palmer Penguins\n")
    fh.write("Source: CC0, used for teaching.\n")

with note.open(encoding="utf-8") as fh:
    text = fh.read()

print(text)
print("characters:", len(text))

### CSV with the standard library

`csv.DictWriter` maps dictionaries to columns, which pairs well with pandas'
`to_dict("records")`. Missing values become empty strings, and the reader returns strings.

In [ ]:
columns = ["species", "island", "bill_length_mm", "body_mass_g"]
sample = penguins[columns].head(10)

sample_path = workdir / "penguins_sample.csv"
with sample_path.open("w", encoding="utf-8", newline="") as fh:
    writer = csv.DictWriter(fh, fieldnames=columns)
    writer.writeheader()
    for row in sample.to_dict("records"):
        writer.writerow(row)

with sample_path.open(encoding="utf-8", newline="") as fh:
    reader = csv.DictReader(fh)
    reloaded = list(reader)

print("bytes on disk:", sample_path.stat().st_size)
print("rows read back:", len(reloaded))
print("first row as strings:", reloaded[0])

Because CSV stores text, `body_mass_g` came back as `'3750.0'`, not a number. That is the reader's
job, or pandas' job. Note also the `newline=""` argument: it stops the `csv` module from adding
blank lines on some platforms.

### JSON with the standard library

JSON round-trips nested Python structures. We compute a small dictionary, write it, then read it
back and compare.

In [ ]:
summary = (
    penguins.dropna(subset=["body_mass_g"])
    .groupby("species")["body_mass_g"]
    .mean()
    .round(1)
    .to_dict()
)
print("in memory:", summary)

summary_path = workdir / "summary.json"
with summary_path.open("w", encoding="utf-8") as fh:
    json.dump(summary, fh, indent=2)

restored = json.loads(summary_path.read_text(encoding="utf-8"))
print("restored  :", restored)
print("round trip equal:", restored == summary)
print("\nfile preview:\n" + summary_path.read_text(encoding="utf-8")[:200])

### pandas CSV and JSON I/O

pandas remembers dtypes and can skip the index. Writing without the index (`index=False`) keeps
the file clean, and reading gives back a numeric column.

In [ ]:
selected = penguins[["species", "island", "body_mass_g"]]
selected_path = workdir / "penguins_selected.csv"
selected.to_csv(selected_path, index=False)

reread = pd.read_csv(selected_path)
print("dtypes after round trip:")
print(reread.dtypes.to_dict())
print("mean mass by species:")
print(reread.groupby("species")["body_mass_g"].mean().round(1))

json_path = workdir / "head.json"
penguins.head(5).to_json(json_path, orient="records", indent=2)
print("\nfrom JSON:")
print(pd.read_json(json_path)[["species", "body_mass_g"]])

### Clean up

We remove the scratch directory on purpose. In a real project the output would go to a known
`data/processed` folder and would be kept, but a teaching notebook should not leave files behind.

In [ ]:
shutil.rmtree(workdir)
print("removed:", not workdir.exists())

## Exercises

Write to a fresh temporary directory in each exercise.

1. **CSV round trip.** Write the first ten penguins to `exercise.csv` using the `csv` module
   with the four columns above, then read them back and print how many rows you recovered.
2. **JSON summary.** Save a dictionary mapping each island to its penguin count as `islands.json`,
   load it again, and assert the loaded value equals the original.
3. **pandas subset.** Use pandas to write `species`, `island`, and `flipper_length_mm` for the
   rows with `species == "Gentoo"` to `gentoo.csv`, then read it back and report the mean
   flipper length.

## Limitations

The `csv` module cannot infer types, so numbers come back as strings unless you convert them;
pandas does infer types but its inference can be wrong for IDs with leading zeros or mixed
columns. JSON has no date type and no comment syntax, so timestamps need a convention
(usually ISO 8601 strings) and configuration files cannot be annotated. None of these examples
handle very large files: `read_csv(..., chunksize=...)` and streaming a line at a time are the
next step when a table does not fit in memory. Finally, writing files is a side effect; a
notebook that only reads is easier to re-run than one that writes into the repository.